# Pizza-Bot - Partie 2 : ordre de jeu et choix de gamme

Mamma-Auto s'est installé le premier : le jeu n'est plus simultané. Il faut aussi choisir une
gamme, Discount ou Premium, sans savoir laquelle Mamma-Auto a retenue.

Marge mensuelle de Pizza-Bot selon sa gamme et le type de Mamma-Auto, en k€ :

| Gamme de Pizza-Bot \ type de Mamma-Auto | Discount | Premium |
|---|---|---|
| **Discount** | 2 | 8 |
| **Premium** | 10 | 3 |

Le service commercial estime à 70 % la probabilité que Mamma-Auto soit Premium.

Règles du projet : Python de base, aucun import.

## Étape 2.1 : qui s'installe en premier ?

In [1]:
EMPLACEMENTS = ["Campus", "Gare"]

GAINS_EMPLACEMENT = {
    ("Campus", "Campus"): (4, 4),
    ("Campus", "Gare"): (9, 7),
    ("Gare", "Campus"): (6, 8),
    ("Gare", "Gare"): (3, 3),
}

# Écrite à l'étape 1.2 : elle sert ici au dernier niveau de l'arbre.
def meilleure_reponse_pizzabot(gains, choix_mamma):
    meilleur = EMPLACEMENTS[0]
    for choix in EMPLACEMENTS:
        if gains[(choix, choix_mamma)][0] > gains[(meilleur, choix_mamma)][0]:
            meilleur = choix
    return meilleur

def induction_a_rebours(gains):
    """Mamma-Auto joue en premier. Renvoie (choix_mamma, choix_pizzabot, gains_de_la_case)."""
    meilleur_choix_mamma = None
    reponse_retenue = None
    case_retenue = None

    for choix_mamma in EMPLACEMENTS:
        # Dernier noeud de l'arbre : Pizza-Bot voit le choix de Mamma-Auto et prend le meilleur pour lui.
        reponse = meilleure_reponse_pizzabot(gains, choix_mamma)
        case = gains[(reponse, choix_mamma)]

        # Racine de l'arbre : Mamma-Auto garde la branche qui lui rapporte le plus (second du couple).
        if case_retenue is None or case[1] > case_retenue[1]:
            meilleur_choix_mamma = choix_mamma
            reponse_retenue = reponse
            case_retenue = case

    return (meilleur_choix_mamma, reponse_retenue, case_retenue)

ma, pb, case = induction_a_rebours(GAINS_EMPLACEMENT)
print(f"Mamma-Auto s'installe : {ma}. Pizza-Bot répond : {pb}. Gains (PB, MA) : {case}")

# --- Vérification : ne rien modifier sous cette ligne ---
dilemme = {("Campus", "Campus"): (3, 3), ("Campus", "Gare"): (0, 5),
           ("Gare", "Campus"): (5, 0), ("Gare", "Gare"): (1, 1)}
assert induction_a_rebours(dilemme) == ("Gare", "Gare", (1, 1)), \
    "au Campus, Mamma-Auto finirait avec 0 (Pizza-Bot répondrait Gare) ; à la Gare, il obtient 1"
print("Induction à rebours validée")

Mamma-Auto s'installe : Campus. Pizza-Bot répond : Gare. Gains (PB, MA) : (6, 8)
Induction à rebours validée


**Lecture du résultat.** Mamma-Auto s'installe au Campus et Pizza-Bot répond par la Gare,
pour des gains de 6 et 8. C'est bien la branche trouvée au brouillon : Mamma-Auto anticipe que
Pizza-Bot ira à la Gare s'il prend le Campus (8 pour lui), et au Campus s'il prend la Gare
(7 pour lui). Il compare 8 et 7, donc il choisit le Campus. L'avantage du premier joueur coûte
3 k€ par mois à Pizza-Bot, qui obtient 6 au lieu des 9 de son équilibre préféré.

## Étape 2.2 : la gamme, contre un adversaire inconnu

In [2]:
GAMMES = ["Discount", "Premium"]

GAINS_GAMME = {
    "Discount": {"Discount": 2, "Premium": 8},
    "Premium": {"Discount": 10, "Premium": 3},
}
CROYANCE = {"Discount": 0.3, "Premium": 0.7}

def esperance_gain(gains, croyance, gamme):
    """Espérance de gain de Pizza-Bot s'il choisit `gamme`, selon sa croyance sur le type de Mamma-Auto."""
    # Pour chaque type possible : probabilité du type x gain contre ce type, puis on additionne.
    total = 0
    for type_mamma in croyance:
        total = total + croyance[type_mamma] * gains[gamme][type_mamma]
    return total

def meilleure_gamme(gains, croyance):
    """Le couple (gamme, espérance) de la gamme d'espérance maximale."""
    gamme_retenue = None
    esperance_retenue = None
    for gamme in GAMMES:
        esperance = esperance_gain(gains, croyance, gamme)
        if esperance_retenue is None or esperance > esperance_retenue:
            gamme_retenue = gamme
            esperance_retenue = esperance
    return (gamme_retenue, esperance_retenue)

for g in GAMMES:
    print(f"E[gain | {g:8}] = {esperance_gain(GAINS_GAMME, CROYANCE, g):.2f} k€/mois")
print("Décision :", meilleure_gamme(GAINS_GAMME, CROYANCE))

# --- Vérification : ne rien modifier sous cette ligne ---
pile = {"Discount": {"Discount": 0, "Premium": 10}, "Premium": {"Discount": 4, "Premium": 4}}
moitie = {"Discount": 0.5, "Premium": 0.5}
assert abs(esperance_gain(pile, moitie, "Discount") - 5) < 1e-9, "0,5 × 0 + 0,5 × 10 = 5"
assert abs(esperance_gain(pile, moitie, "Premium") - 4) < 1e-9, "un gain certain a pour espérance lui-même"
gamme, e = meilleure_gamme(pile, moitie)
assert gamme == "Discount" and abs(e - 5) < 1e-9, "5 l'emporte sur 4"
assert meilleure_gamme(pile, {"Discount": 0.9, "Premium": 0.1})[0] == "Premium", "la croyance change la décision"
print("Espérance validée")

E[gain | Discount] = 6.20 k€/mois
E[gain | Premium ] = 5.10 k€/mois
Décision : ('Discount', 6.199999999999999)
Espérance validée


**Lecture du résultat.** Discount rapporte 6,20 k€ par mois en moyenne, contre 5,10 pour
Premium : c'est Discount qu'il faut choisir. Ces deux nombres sont ceux du brouillon,
0,3 x 2 + 0,7 x 8 et 0,3 x 10 + 0,7 x 3. Le raisonnement tient parce que l'adversaire inconnu
est remplacé par une loterie entre deux adversaires connus.

## Étape 2.3 : la décision est-elle robuste ?

In [3]:
GAMMES = ["Discount", "Premium"]

GAINS_GAMME = {
    "Discount": {"Discount": 2, "Premium": 8},
    "Premium": {"Discount": 10, "Premium": 3},
}

def esperance_gain(gains, croyance, gamme):
    return sum(croyance[t] * gains[gamme][t] for t in croyance)

def croyance_de_bascule(gains, nb_pas=1000):
    """La première valeur de c = P(Premium) pour laquelle Discount rapporte au moins autant que Premium."""
    # On fait tourner un entier i, et on calcule c = i / nb_pas : pas d'accumulation d'erreurs.
    for i in range(nb_pas + 1):
        c = i / nb_pas
        croyance = {"Premium": c, "Discount": 1 - c}
        if esperance_gain(gains, croyance, "Discount") >= esperance_gain(gains, croyance, "Premium"):
            return c
    return None

print(f"Discount devient le meilleur choix dès P(Premium) = {croyance_de_bascule(GAINS_GAMME)}")

# --- Vérification : ne rien modifier sous cette ligne ---
pile = {"Discount": {"Discount": 0, "Premium": 10}, "Premium": {"Discount": 4, "Premium": 4}}
assert abs(croyance_de_bascule(pile) - 0.4) < 0.002, "10c = 4 donne c = 0,4"
jamais = {"Discount": {"Discount": 0, "Premium": 1}, "Premium": {"Discount": 5, "Premium": 5}}
assert croyance_de_bascule(jamais) is None, "Discount ne rattrape jamais un gain certain de 5"
print("Bascule validée")

Discount devient le meilleur choix dès P(Premium) = 0.616
Bascule validée


**Lecture du résultat.** La décision bascule à partir de 0,616, alors que le brouillon donne
8/13, soit 0,6154 : le balayage avance par pas de un millième et renvoie la première valeur qui
franchit la limite. L'estimation du service commercial vaut 0,7, donc il ne reste que huit points
de marge avant que Premium redevienne le meilleur choix.

## Bonus : la valeur de l'information parfaite

Une étude de marché donnerait le type de Mamma-Auto avant le choix de la gamme. On choisirait
alors la meilleure gamme contre chaque type.

In [4]:
GAMMES = ["Discount", "Premium"]

GAINS_GAMME = {
    "Discount": {"Discount": 2, "Premium": 8},
    "Premium": {"Discount": 10, "Premium": 3},
}
CROYANCE = {"Discount": 0.3, "Premium": 0.7}

def esperance_gain(gains, croyance, gamme):
    total = 0
    for type_mamma in croyance:
        total = total + croyance[type_mamma] * gains[gamme][type_mamma]
    return total

# Avec l'information : contre chaque type, on prend la gamme qui rapporte le plus.
esperance_avec_info = 0
for type_mamma in CROYANCE:
    meilleur_gain = None
    for gamme in GAMMES:
        gain = GAINS_GAMME[gamme][type_mamma]
        if meilleur_gain is None or gain > meilleur_gain:
            meilleur_gain = gain
    print(f"Si Mamma-Auto est {type_mamma:8}, le meilleur gain possible est {meilleur_gain} k€")
    esperance_avec_info = esperance_avec_info + CROYANCE[type_mamma] * meilleur_gain

# Sans information : la meilleure des deux espérances.
esperance_sans_info = max(esperance_gain(GAINS_GAMME, CROYANCE, "Discount"),
                          esperance_gain(GAINS_GAMME, CROYANCE, "Premium"))

print(f"Espérance avec information : {esperance_avec_info:.2f} k€/mois")
print(f"Espérance sans information : {esperance_sans_info:.2f} k€/mois")
print(f"Valeur de l'information parfaite : {esperance_avec_info - esperance_sans_info:.2f} k€/mois")

Si Mamma-Auto est Discount, le meilleur gain possible est 10 k€
Si Mamma-Auto est Premium , le meilleur gain possible est 8 k€
Espérance avec information : 8.60 k€/mois
Espérance sans information : 6.20 k€/mois
Valeur de l'information parfaite : 2.40 k€/mois


**Lecture du résultat.** Connaître le type de Mamma-Auto vaut 2,40 k€ par mois : on passerait
de 6,20 à 8,60. C'est le prix maximal qu'il est rationnel de payer pour une étude de marché qui
lèverait complètement le doute. Au-delà, l'étude coûte plus cher que ce qu'elle rapporte.